# Self Supervised and Contrastive Representation Learning on STL10

## Part A: Supervised Learning with Limited Labels




In [1]:
# Step 0: Personal parameters

SID4 = 2046
SEED = SID4
SLICE = SID4 % 1000
HP_ID = SID4 % 6
CLS_A = SID4 % 10
CLS_B = (CLS_A + 1 + ((SID4 // 10) % 9)) % 10

print("SID4:", SID4)
print("SEED:", SEED)
print("SLICE:", SLICE)
print("HP_ID:", HP_ID)
print("CLS_A:", CLS_A)
print("CLS_B:", CLS_B)


SID4: 2046
SEED: 2046
SLICE: 46
HP_ID: 0
CLS_A: 6
CLS_B: 3


## Environment and Reproducibility

In [2]:
import os
import random
import json
import time
import platform
from datetime import datetime

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from torchvision.models import resnet18

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("Reproducibility seed:", SEED)


Python: 3.13.15
PyTorch: 2.11.0+cu130
Device: cuda
GPU: Tesla T4
Reproducibility seed: 2046


## Load STL10




In [3]:
DATA_ROOT = "./data"

STL10_MEAN = (0.4467, 0.4398, 0.4066)
STL10_STD = (0.2603, 0.2566, 0.2713)

train_transform = transforms.Compose([
    transforms.RandomCrop(
        96,
        padding=12,
        padding_mode="reflect",
    ),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        STL10_MEAN,
        STL10_STD,
    ),
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        STL10_MEAN,
        STL10_STD,
    ),
])

train_full = datasets.STL10(
    root=DATA_ROOT,
    split="train",
    download=True,
    transform=train_transform,
)

test_dataset = datasets.STL10(
    root=DATA_ROOT,
    split="test",
    download=True,
    transform=test_transform,
)

print("Full labeled train size:", len(train_full))
print("Test size:", len(test_dataset))

assert len(train_full) == 5000
assert len(test_dataset) == 8000


100%|██████████| 2.64G/2.64G [02:50<00:00, 15.5MB/s]


Full labeled train size: 5000
Test size: 8000


## Select the Required 500 Labeled Images

In [4]:
LABELED_SUBSET_SIZE = 500

subset_rng = np.random.default_rng(SEED)

labeled_indices = subset_rng.choice(
    len(train_full),
    size=LABELED_SUBSET_SIZE,
    replace=False,
)

labeled_indices = labeled_indices.tolist()

train_subset = Subset(
    train_full,
    labeled_indices,
)

os.makedirs("artifacts", exist_ok=True)

with open(
    "artifacts/labeled_500_indices.json",
    "w",
    encoding="utf8",
) as f:
    json.dump(labeled_indices, f, indent=2)

print("Selected labeled images:", len(train_subset))
print("First 10 selected indices:", labeled_indices[:10])

assert len(train_subset) == 500
assert len(set(labeled_indices)) == 500


Selected labeled images: 500
First 10 selected indices: [633, 1659, 3085, 3851, 2592, 2255, 263, 4290, 1272, 1065]


## Inspect the Limited Label Subset



In [5]:
selected_targets = np.array(
    train_full.labels
)[labeled_indices]

class_counts = pd.Series(
    selected_targets
).value_counts().sort_index()

class_distribution = pd.DataFrame({
    "Class ID": list(range(10)),
    "Class Name": train_full.classes,
    "Count": [
        int(class_counts.get(i, 0))
        for i in range(10)
    ],
})

class_distribution


,Class ID,Class Name,Count
0,0,airplane,47
1,1,bird,54
2,2,car,50
3,3,cat,43
4,4,deer,51
5,5,dog,48
6,6,horse,43
7,7,monkey,62
8,8,ship,51
9,9,truck,51


## Data Loaders

In [6]:
BATCH_SIZE = 64
NUM_WORKERS = 2

train_generator = torch.Generator()
train_generator.manual_seed(SEED)

train_loader = DataLoader(
    train_subset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    generator=train_generator,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=128,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

images, labels = next(iter(train_loader))

print("Training batch image shape:", images.shape)
print("Training batch label shape:", labels.shape)
print("Label range:", int(labels.min()), "to", int(labels.max()))

assert images.shape[1:] == (3, 96, 96)
assert labels.min() >= 0
assert labels.max() <= 9


Training batch image shape: torch.Size([64, 3, 96, 96])
Training batch label shape: torch.Size([64])
Label range: 0 to 9


## Part A Model: ResNet18 from Scratch




In [7]:
supervised_model = resnet18(
    weights=None
)

supervised_model.fc = nn.Linear(
    supervised_model.fc.in_features,
    10,
)

supervised_model = supervised_model.to(device)

total_params = sum(
    p.numel()
    for p in supervised_model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in supervised_model.parameters()
    if p.requires_grad
)

print("Architecture: ResNet18")
print("Pretrained weights: None")
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

assert total_params == trainable_params


Architecture: ResNet18
Pretrained weights: None
Total parameters: 11,181,642
Trainable parameters: 11,181,642


## Training Setup

In [8]:
NUM_EPOCHS = 12
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    supervised_model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=NUM_EPOCHS,
)

print("Epochs:", NUM_EPOCHS)
print("Optimizer: Adam")
print("Learning rate:", LEARNING_RATE)
print("Weight decay:", WEIGHT_DECAY)
print("Loss: CrossEntropyLoss")


Epochs: 12
Optimizer: Adam
Learning rate: 0.001
Weight decay: 0.0001
Loss: CrossEntropyLoss


## Train the Supervised Baseline

In [9]:
def evaluate_accuracy(model, loader):
    model.eval()

    correct = 0
    total = 0
    total_loss = 0.0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(
                device,
                non_blocking=True,
            )
            labels = labels.to(
                device,
                non_blocking=True,
            )

            logits = model(images)
            loss = criterion(logits, labels)

            total_loss += (
                float(loss.detach().cpu())
                * labels.size(0)
            )

            predictions = logits.argmax(dim=1)

            correct += int(
                (predictions == labels).sum().item()
            )
            total += labels.size(0)

    return {
        "loss": total_loss / total,
        "accuracy": correct / total,
    }


In [10]:
training_history = []
training_start = time.time()

for epoch in range(1, NUM_EPOCHS + 1):
    supervised_model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for batch_index, (images, labels) in enumerate(
        train_loader,
        start=1,
    ):
        images = images.to(
            device,
            non_blocking=True,
        )
        labels = labels.to(
            device,
            non_blocking=True,
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        logits = supervised_model(images)
        loss = criterion(logits, labels)

        if not torch.isfinite(loss):
            raise RuntimeError(
                f"Nonfinite loss at epoch "
                f"{epoch}, batch {batch_index}"
            )

        loss.backward()
        optimizer.step()

        running_loss += (
            float(loss.detach().cpu())
            * labels.size(0)
        )

        predictions = logits.argmax(dim=1)

        correct += int(
            (predictions == labels).sum().item()
        )
        total += labels.size(0)

    scheduler.step()

    epoch_loss = running_loss / total
    epoch_accuracy = correct / total

    training_history.append({
        "epoch": epoch,
        "train_loss": epoch_loss,
        "train_accuracy": epoch_accuracy,
        "learning_rate": scheduler.get_last_lr()[0],
    })

    print(
        f"Epoch {epoch:02d}/{NUM_EPOCHS} | "
        f"loss={epoch_loss:.4f} | "
        f"train_acc={epoch_accuracy * 100:.2f}% | "
        f"lr={scheduler.get_last_lr()[0]:.6f}"
    )

training_time = time.time() - training_start

print()
print(
    f"Training time: "
    f"{training_time:.2f} seconds"
)


Epoch 01/12 | loss=2.2179 | train_acc=20.80% | lr=0.000983
Epoch 02/12 | loss=1.9523 | train_acc=28.00% | lr=0.000933
Epoch 03/12 | loss=1.7701 | train_acc=32.80% | lr=0.000854
Epoch 04/12 | loss=1.4591 | train_acc=45.20% | lr=0.000750
Epoch 05/12 | loss=1.3794 | train_acc=50.20% | lr=0.000629
Epoch 06/12 | loss=1.2625 | train_acc=53.60% | lr=0.000500
Epoch 07/12 | loss=1.1458 | train_acc=60.80% | lr=0.000371
Epoch 08/12 | loss=1.0290 | train_acc=64.00% | lr=0.000250
Epoch 09/12 | loss=0.9866 | train_acc=67.80% | lr=0.000146
Epoch 10/12 | loss=0.8550 | train_acc=71.80% | lr=0.000067
Epoch 11/12 | loss=0.8171 | train_acc=72.40% | lr=0.000017
Epoch 12/12 | loss=0.7544 | train_acc=75.40% | lr=0.000000

Training time: 11.11 seconds


## Training History

In [11]:
history_df = pd.DataFrame(
    training_history
)

history_df


,epoch,train_loss,train_accuracy,learning_rate
0,1,2.217904,0.208,0.000983
1,2,1.952276,0.280,0.000933
2,3,1.770098,0.328,0.000854
3,4,1.459105,0.452,0.000750
4,5,1.379353,0.502,0.000629
5,6,1.262508,0.536,0.000500
6,7,1.145843,0.608,0.000371
7,8,1.029045,0.640,0.000250
8,9,0.986600,0.678,0.000146
9,10,0.854977,0.718,0.000067


## Part A Test Accuracy

In [12]:
test_metrics = evaluate_accuracy(
    supervised_model,
    test_loader,
)

test_accuracy = test_metrics["accuracy"]
test_loss = test_metrics["loss"]

print(
    f"Test loss: "
    f"{test_loss:.4f}"
)
print(
    f"Test accuracy: "
    f"{test_accuracy * 100:.2f}%"
)

assert np.isfinite(test_loss)
assert 0.0 <= test_accuracy <= 1.0


Test loss: 1.5480
Test accuracy: 44.40%


## Part A Interpretation

The supervised model was trained end to end from random initialization using only 500 labeled STL10 images. The measured test accuracy above is the supervised limited label baseline that the Rotation SSL and SimCLR representations will be compared against later.


## Save Checkpoint and Part A Artifacts

In [13]:
os.makedirs(
    "checkpoints",
    exist_ok=True,
)

checkpoint_path = (
    "checkpoints/supervised_resnet18.pt"
)

torch.save(
    {
        "model_state_dict": (
            supervised_model.state_dict()
        ),
        "architecture": "resnet18",
        "num_classes": 10,
        "pretrained": False,
        "SID4": SID4,
        "SEED": SEED,
        "labeled_indices": labeled_indices,
        "epochs": NUM_EPOCHS,
        "test_accuracy": test_accuracy,
    },
    checkpoint_path,
)

history_df.to_csv(
    "artifacts/part_a_training_history.csv",
    index=False,
)

part_a_metrics = {
    "timestamp": datetime.now().isoformat(),
    "SID4": SID4,
    "SEED": SEED,
    "SLICE": SLICE,
    "HP_ID": HP_ID,
    "CLS_A": CLS_A,
    "CLS_B": CLS_B,
    "dataset": "STL10",
    "architecture": "ResNet18",
    "pretrained_weights": False,
    "labeled_training_images": 500,
    "epochs": NUM_EPOCHS,
    "batch_size": BATCH_SIZE,
    "optimizer": "Adam",
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "test_loss": test_loss,
    "test_accuracy": test_accuracy,
    "training_time_seconds": training_time,
}

with open(
    "artifacts/part_a_metrics.json",
    "w",
    encoding="utf8",
) as f:
    json.dump(
        part_a_metrics,
        f,
        indent=2,
    )

print(
    json.dumps(
        part_a_metrics,
        indent=2,
    )
)

print()
print(
    "Checkpoint saved:",
    checkpoint_path,
)


{
  "timestamp": "2026-10-06T18:46:13.772382",
  "SID4": 2046,
  "SEED": 2046,
  "SLICE": 46,
  "HP_ID": 0,
  "CLS_A": 6,
  "CLS_B": 3,
  "dataset": "STL10",
  "architecture": "ResNet18",
  "pretrained_weights": false,
  "labeled_training_images": 500,
  "epochs": 12,
  "batch_size": 64,
  "optimizer": "Adam",
  "learning_rate": 0.001,
  "weight_decay": 0.0001,
  "test_loss": 1.548024697303772,
  "test_accuracy": 0.444,
  "training_time_seconds": 11.11295747756958
}

Checkpoint saved: checkpoints/supervised_resnet18.pt


# Part B: Rotation Self Supervised Learning

Use all 100,000 unlabeled STL10 images to train a fresh ResNet18 to predict rotations of 0, 90, 180, or 270 degrees. Then remove the rotation head, freeze the encoder, and train only a linear classifier using the exact same 500 labeled images from Part A.


In [14]:
unlabeled_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(STL10_MEAN, STL10_STD),
])

unlabeled_dataset = datasets.STL10(
    root=DATA_ROOT,
    split="unlabeled",
    download=True,
    transform=unlabeled_transform,
)

print("Unlabeled images:", len(unlabeled_dataset))
assert len(unlabeled_dataset) == 100000


Unlabeled images: 100000


In [15]:
class RotationDataset(torch.utils.data.Dataset):
    def __init__(self, dataset, seed):
        self.dataset = dataset
        self.seed = seed

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, index):
        image, _ = self.dataset[index]
        rng = random.Random(self.seed + index)
        rotation_class = rng.randrange(4)
        image = torch.rot90(
            image,
            k=rotation_class,
            dims=(1, 2),
        )
        return image, rotation_class


rotation_dataset = RotationDataset(
    unlabeled_dataset,
    SEED,
)

image, rotation_label = rotation_dataset[0]

print("Image shape:", image.shape)
print("Rotation class:", rotation_label)

assert image.shape == (3, 96, 96)
assert rotation_label in [0, 1, 2, 3]


Image shape: torch.Size([3, 96, 96])
Rotation class: 3


In [16]:
ROTATION_BATCH_SIZE = 256
ROTATION_EPOCHS = 15
ROTATION_LR = 1e-3
ROTATION_WEIGHT_DECAY = 1e-4

rotation_generator = torch.Generator().manual_seed(SEED)

rotation_loader = DataLoader(
    rotation_dataset,
    batch_size=ROTATION_BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    generator=rotation_generator,
)

print("Batches per epoch:", len(rotation_loader))
print("Epochs:", ROTATION_EPOCHS)


Batches per epoch: 391
Epochs: 15


In [17]:
rotation_model = resnet18(weights=None)
rotation_feature_dim = rotation_model.fc.in_features
rotation_model.fc = nn.Linear(rotation_feature_dim, 4)
rotation_model = rotation_model.to(device)

rotation_total_params = sum(
    p.numel() for p in rotation_model.parameters()
)
rotation_trainable_params = sum(
    p.numel()
    for p in rotation_model.parameters()
    if p.requires_grad
)

print("Architecture: ResNet18")
print("Pretrained weights: None")
print("Rotation classes: 4")
print("Total parameters:", f"{rotation_total_params:,}")
print("Trainable parameters:", f"{rotation_trainable_params:,}")


Architecture: ResNet18
Pretrained weights: None
Rotation classes: 4
Total parameters: 11,178,564
Trainable parameters: 11,178,564


In [18]:
rotation_criterion = nn.CrossEntropyLoss()

rotation_optimizer = torch.optim.Adam(
    rotation_model.parameters(),
    lr=ROTATION_LR,
    weight_decay=ROTATION_WEIGHT_DECAY,
)

rotation_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    rotation_optimizer,
    T_max=ROTATION_EPOCHS,
)


In [19]:
rotation_history = []
rotation_start = time.time()

for epoch in range(1, ROTATION_EPOCHS + 1):
    rotation_model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for batch_index, (images, rotation_labels) in enumerate(
        rotation_loader,
        start=1,
    ):
        images = images.to(device, non_blocking=True)
        rotation_labels = rotation_labels.to(
            device,
            non_blocking=True,
        )

        rotation_optimizer.zero_grad(set_to_none=True)

        logits = rotation_model(images)
        loss = rotation_criterion(logits, rotation_labels)

        if not torch.isfinite(loss):
            raise RuntimeError(
                f"Nonfinite rotation loss at epoch {epoch}, "
                f"batch {batch_index}"
            )

        loss.backward()
        rotation_optimizer.step()

        running_loss += float(loss.detach().cpu()) * rotation_labels.size(0)
        predictions = logits.argmax(dim=1)
        correct += int((predictions == rotation_labels).sum().item())
        total += rotation_labels.size(0)

    rotation_scheduler.step()

    epoch_loss = running_loss / total
    epoch_accuracy = correct / total

    rotation_history.append({
        "epoch": epoch,
        "rotation_loss": epoch_loss,
        "rotation_accuracy": epoch_accuracy,
        "learning_rate": rotation_scheduler.get_last_lr()[0],
    })

    print(
        f"Rotation epoch {epoch:02d}/{ROTATION_EPOCHS} | "
        f"loss={epoch_loss:.4f} | "
        f"accuracy={epoch_accuracy * 100:.2f}% | "
        f"lr={rotation_scheduler.get_last_lr()[0]:.6f}"
    )

rotation_training_time = time.time() - rotation_start

print()
print(f"Rotation pretraining time: {rotation_training_time:.2f} seconds")


Rotation epoch 01/15 | loss=0.9341 | accuracy=61.32% | lr=0.000989
Rotation epoch 02/15 | loss=0.7271 | accuracy=71.11% | lr=0.000957
Rotation epoch 03/15 | loss=0.6287 | accuracy=75.49% | lr=0.000905
Rotation epoch 04/15 | loss=0.5532 | accuracy=78.54% | lr=0.000835
Rotation epoch 05/15 | loss=0.4841 | accuracy=81.52% | lr=0.000750
Rotation epoch 06/15 | loss=0.4136 | accuracy=84.47% | lr=0.000655
Rotation epoch 07/15 | loss=0.3349 | accuracy=87.50% | lr=0.000552
Rotation epoch 08/15 | loss=0.2434 | accuracy=91.07% | lr=0.000448
Rotation epoch 09/15 | loss=0.1401 | accuracy=94.97% | lr=0.000345
Rotation epoch 10/15 | loss=0.0695 | accuracy=97.58% | lr=0.000250
Rotation epoch 11/15 | loss=0.0263 | accuracy=99.17% | lr=0.000165
Rotation epoch 12/15 | loss=0.0079 | accuracy=99.79% | lr=0.000095
Rotation epoch 13/15 | loss=0.0018 | accuracy=99.98% | lr=0.000043
Rotation epoch 14/15 | loss=0.0008 | accuracy=100.00% | lr=0.000011
Rotation epoch 15/15 | loss=0.0006 | accuracy=100.00% | lr=0.

In [20]:
rotation_history_df = pd.DataFrame(rotation_history)
rotation_history_df


,epoch,rotation_loss,rotation_accuracy,learning_rate
0,1,0.934053,0.61316,0.000989
1,2,0.727063,0.71106,0.000957
2,3,0.628740,0.75486,0.000905
3,4,0.553174,0.78536,0.000835
4,5,0.484110,0.81515,0.000750
5,6,0.413575,0.84469,0.000655
6,7,0.334891,0.87498,0.000552
7,8,0.243404,0.91070,0.000448
8,9,0.140107,0.94974,0.000345
9,10,0.069482,0.97585,0.000250


In [21]:
os.makedirs("checkpoints", exist_ok=True)

rotation_checkpoint_path = "checkpoints/rotation_resnet18.pt"

torch.save(
    {
        "model_state_dict": rotation_model.state_dict(),
        "architecture": "resnet18",
        "pretrained": False,
        "rotation_classes": 4,
        "feature_dim": rotation_feature_dim,
        "SID4": SID4,
        "SEED": SEED,
        "epochs": ROTATION_EPOCHS,
    },
    rotation_checkpoint_path,
)

rotation_history_df.to_csv(
    "artifacts/rotation_training_history.csv",
    index=False,
)

print("Saved:", rotation_checkpoint_path)


Saved: checkpoints/rotation_resnet18.pt


## Linear Evaluation

Replace the rotation head with a ten class classifier. Freeze the entire encoder and allow only the new linear layer to update.


In [22]:
rotation_model.fc = nn.Linear(
    rotation_feature_dim,
    10,
)

for parameter in rotation_model.parameters():
    parameter.requires_grad = False

for parameter in rotation_model.fc.parameters():
    parameter.requires_grad = True

rotation_model = rotation_model.to(device)

encoder_trainable = sum(
    p.numel()
    for name, p in rotation_model.named_parameters()
    if p.requires_grad and not name.startswith("fc.")
)

linear_trainable = sum(
    p.numel()
    for p in rotation_model.fc.parameters()
    if p.requires_grad
)

print("Trainable encoder parameters:", encoder_trainable)
print("Trainable linear parameters:", linear_trainable)

assert encoder_trainable == 0
assert linear_trainable > 0

print("Freeze check passed.")


Trainable encoder parameters: 0
Trainable linear parameters: 5130
Freeze check passed.


In [23]:
linear_train_full = datasets.STL10(
    root=DATA_ROOT,
    split="train",
    download=False,
    transform=test_transform,
)

linear_train_subset = Subset(
    linear_train_full,
    labeled_indices,
)

linear_generator = torch.Generator().manual_seed(SEED)

linear_train_loader = DataLoader(
    linear_train_subset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    generator=linear_generator,
)

print("Linear evaluation labeled images:", len(linear_train_subset))
assert len(linear_train_subset) == 500


Linear evaluation labeled images: 500


In [24]:
LINEAR_EPOCHS = 20
LINEAR_LR = 1e-3

linear_criterion = nn.CrossEntropyLoss()

linear_optimizer = torch.optim.Adam(
    rotation_model.fc.parameters(),
    lr=LINEAR_LR,
)

linear_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    linear_optimizer,
    T_max=LINEAR_EPOCHS,
)


In [25]:
linear_history = []
linear_start = time.time()

for epoch in range(1, LINEAR_EPOCHS + 1):
    rotation_model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in linear_train_loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        linear_optimizer.zero_grad(set_to_none=True)

        logits = rotation_model(images)
        loss = linear_criterion(logits, labels)

        if not torch.isfinite(loss):
            raise RuntimeError(
                f"Nonfinite linear loss at epoch {epoch}"
            )

        loss.backward()
        linear_optimizer.step()

        running_loss += float(loss.detach().cpu()) * labels.size(0)
        predictions = logits.argmax(dim=1)
        correct += int((predictions == labels).sum().item())
        total += labels.size(0)

    linear_scheduler.step()

    epoch_loss = running_loss / total
    epoch_accuracy = correct / total

    linear_history.append({
        "epoch": epoch,
        "train_loss": epoch_loss,
        "train_accuracy": epoch_accuracy,
        "learning_rate": linear_scheduler.get_last_lr()[0],
    })

    print(
        f"Linear epoch {epoch:02d}/{LINEAR_EPOCHS} | "
        f"loss={epoch_loss:.4f} | "
        f"train_acc={epoch_accuracy * 100:.2f}% | "
        f"lr={linear_scheduler.get_last_lr()[0]:.6f}"
    )

linear_training_time = time.time() - linear_start
print()
print(f"Linear training time: {linear_training_time:.2f} seconds")


Linear epoch 01/20 | loss=2.3018 | train_acc=11.20% | lr=0.000994
Linear epoch 02/20 | loss=2.2098 | train_acc=20.00% | lr=0.000976
Linear epoch 03/20 | loss=2.1572 | train_acc=22.40% | lr=0.000946
Linear epoch 04/20 | loss=2.1201 | train_acc=26.60% | lr=0.000905
Linear epoch 05/20 | loss=2.0944 | train_acc=26.80% | lr=0.000854
Linear epoch 06/20 | loss=2.0748 | train_acc=29.20% | lr=0.000794
Linear epoch 07/20 | loss=2.0534 | train_acc=31.80% | lr=0.000727
Linear epoch 08/20 | loss=2.0382 | train_acc=32.20% | lr=0.000655
Linear epoch 09/20 | loss=2.0214 | train_acc=35.40% | lr=0.000578
Linear epoch 10/20 | loss=2.0108 | train_acc=35.40% | lr=0.000500
Linear epoch 11/20 | loss=1.9989 | train_acc=36.60% | lr=0.000422
Linear epoch 12/20 | loss=1.9917 | train_acc=38.00% | lr=0.000345
Linear epoch 13/20 | loss=1.9838 | train_acc=37.00% | lr=0.000273
Linear epoch 14/20 | loss=1.9770 | train_acc=36.20% | lr=0.000206
Linear epoch 15/20 | loss=1.9719 | train_acc=35.00% | lr=0.000146
Linear epo

In [26]:
rotation_model.eval()

correct = 0
total = 0
total_loss = 0.0

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        logits = rotation_model(images)
        loss = linear_criterion(logits, labels)

        total_loss += float(loss.detach().cpu()) * labels.size(0)
        predictions = logits.argmax(dim=1)
        correct += int((predictions == labels).sum().item())
        total += labels.size(0)

rotation_test_loss = total_loss / total
rotation_test_accuracy = correct / total

print(f"Rotation SSL test loss: {rotation_test_loss:.4f}")
print(f"Rotation SSL test accuracy: {rotation_test_accuracy * 100:.2f}%")

assert np.isfinite(rotation_test_loss)
assert 0 <= rotation_test_accuracy <= 1


Rotation SSL test loss: 2.0612
Rotation SSL test accuracy: 28.86%


In [27]:
part_ab_comparison = pd.DataFrame([
    {
        "Method": "Supervised limited labels",
        "Labeled Images": 500,
        "Unlabeled Pretraining Images": 0,
        "Test Accuracy": test_accuracy,
    },
    {
        "Method": "Rotation SSL + linear evaluation",
        "Labeled Images": 500,
        "Unlabeled Pretraining Images": 100000,
        "Test Accuracy": rotation_test_accuracy,
    },
])

part_ab_comparison


,Method,Labeled Images,Unlabeled Pretraining Images,Test Accuracy
0,Supervised limited labels,500,0,0.444000
1,Rotation SSL + linear evaluation,500,100000,0.288625


## Part B Interpretation

The rotation encoder was trained on unlabeled STL10 images without object class labels. During linear evaluation, the encoder remained frozen and only the new classifier was trained on the same 500 labeled images from Part A. The measured test accuracy above will later be compared with SimCLR.


In [28]:
linear_history_df = pd.DataFrame(linear_history)

linear_history_df.to_csv(
    "artifacts/rotation_linear_history.csv",
    index=False,
)

rotation_linear_checkpoint = (
    "checkpoints/rotation_linear_classifier.pt"
)

torch.save(
    {
        "model_state_dict": rotation_model.state_dict(),
        "architecture": "resnet18",
        "encoder_frozen": True,
        "linear_classes": 10,
        "labeled_indices": labeled_indices,
        "test_accuracy": rotation_test_accuracy,
        "SID4": SID4,
        "SEED": SEED,
    },
    rotation_linear_checkpoint,
)

part_b_metrics = {
    "timestamp": datetime.now().isoformat(),
    "SID4": SID4,
    "SEED": SEED,
    "unlabeled_images": len(unlabeled_dataset),
    "rotation_epochs": ROTATION_EPOCHS,
    "rotation_final_loss": rotation_history[-1]["rotation_loss"],
    "rotation_final_accuracy": rotation_history[-1]["rotation_accuracy"],
    "rotation_training_time_seconds": rotation_training_time,
    "linear_labeled_images": 500,
    "linear_epochs": LINEAR_EPOCHS,
    "linear_trainable_encoder_parameters": encoder_trainable,
    "linear_trainable_classifier_parameters": linear_trainable,
    "test_loss": rotation_test_loss,
    "test_accuracy": rotation_test_accuracy,
    "linear_training_time_seconds": linear_training_time,
}

with open(
    "artifacts/part_b_metrics.json",
    "w",
    encoding="utf8",
) as f:
    json.dump(part_b_metrics, f, indent=2)

print(json.dumps(part_b_metrics, indent=2))
print()
print("Saved:", rotation_linear_checkpoint)


{
  "timestamp": "2026-10-06T19:07:05.450065",
  "SID4": 2046,
  "SEED": 2046,
  "unlabeled_images": 100000,
  "rotation_epochs": 15,
  "rotation_final_loss": 0.0005854049654584378,
  "rotation_final_accuracy": 1.0,
  "rotation_training_time_seconds": 1218.2565331459045,
  "linear_labeled_images": 500,
  "linear_epochs": 20,
  "linear_trainable_encoder_parameters": 0,
  "linear_trainable_classifier_parameters": 5130,
  "test_loss": 2.0611600646972654,
  "test_accuracy": 0.288625,
  "linear_training_time_seconds": 11.256187200546265
}

Saved: checkpoints/rotation_linear_classifier.pt


In [29]:
# Part C: SimCLR setup

SIMCLR_SUBSET_SIZE = 20000
SIMCLR_EPOCHS = 15
SIMCLR_BATCH_SIZE = 256
SIMCLR_LR = 1e-3
SIMCLR_WEIGHT_DECAY = 1e-4
SIMCLR_TAU = 0.2
PROJECTION_DIM = 128

simclr_rng = np.random.default_rng(SEED)

simclr_indices = simclr_rng.choice(
    len(unlabeled_dataset),
    size=SIMCLR_SUBSET_SIZE,
    replace=False,
).tolist()

print("SimCLR unlabeled subset:", len(simclr_indices))
print("Temperature tau:", SIMCLR_TAU)
print("First 10 subset indices:", simclr_indices[:10])

assert len(simclr_indices) == 20000
assert len(set(simclr_indices)) == 20000

SimCLR unlabeled subset: 20000
Temperature tau: 0.2
First 10 subset indices: [81786, 4025, 67351, 98292, 35722, 39806, 13558, 17408, 33328, 65413]


In [30]:
simclr_augmentation = transforms.Compose([
    transforms.RandomResizedCrop(
        96,
        scale=(0.6, 1.0),
    ),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(
        0.4,
        0.4,
        0.4,
        0.1,
    ),
    transforms.RandomGrayscale(p=0.2),
    transforms.ToTensor(),
    transforms.Normalize(
        STL10_MEAN,
        STL10_STD,
    ),
])

print("SimCLR augmentations:")
print("1. RandomResizedCrop")
print("2. RandomHorizontalFlip")
print("3. ColorJitter")
print("4. RandomGrayscale")

SimCLR augmentations:
1. RandomResizedCrop
2. RandomHorizontalFlip
3. ColorJitter
4. RandomGrayscale


In [31]:
class SimCLRDataset(torch.utils.data.Dataset):
    def __init__(self, raw_data, indices, transform):
        self.raw_data = raw_data
        self.indices = indices
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, position):
        original_index = self.indices[position]

        image = self.raw_data[original_index]

        image = np.transpose(
            image,
            (1, 2, 0),
        )

        image = transforms.functional.to_pil_image(
            image
        )

        view1 = self.transform(image)
        view2 = self.transform(image)

        return view1, view2


simclr_dataset = SimCLRDataset(
    unlabeled_dataset.data,
    simclr_indices,
    simclr_augmentation,
)

view1, view2 = simclr_dataset[0]

print("View 1 shape:", view1.shape)
print("View 2 shape:", view2.shape)

assert view1.shape == (3, 96, 96)
assert view2.shape == (3, 96, 96)

View 1 shape: torch.Size([3, 96, 96])
View 2 shape: torch.Size([3, 96, 96])


In [32]:
simclr_generator = torch.Generator()
simclr_generator.manual_seed(SEED)

simclr_loader = DataLoader(
    simclr_dataset,
    batch_size=SIMCLR_BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    drop_last=True,
    generator=simclr_generator,
)

print("SimCLR batches per epoch:", len(simclr_loader))
print("SimCLR epochs:", SIMCLR_EPOCHS)

SimCLR batches per epoch: 78
SimCLR epochs: 15


In [33]:
class SimCLRModel(nn.Module):
    def __init__(self, projection_dim=128):
        super().__init__()

        self.encoder = resnet18(
            weights=None
        )

        feature_dim = self.encoder.fc.in_features

        self.encoder.fc = nn.Identity()

        self.projector = nn.Sequential(
            nn.Linear(
                feature_dim,
                feature_dim,
            ),
            nn.ReLU(),
            nn.Linear(
                feature_dim,
                projection_dim,
            ),
        )

        self.feature_dim = feature_dim

    def forward(self, x):
        features = self.encoder(x)
        projections = self.projector(features)

        return features, projections


simclr_model = SimCLRModel(
    projection_dim=PROJECTION_DIM
).to(device)

simclr_total_params = sum(
    p.numel()
    for p in simclr_model.parameters()
)

simclr_trainable_params = sum(
    p.numel()
    for p in simclr_model.parameters()
    if p.requires_grad
)

print("Backbone: ResNet18")
print("Pretrained weights: None")
print("Feature dimension:", simclr_model.feature_dim)
print("Projection dimension:", PROJECTION_DIM)
print("Total parameters:", f"{simclr_total_params:,}")
print("Trainable parameters:", f"{simclr_trainable_params:,}")

Backbone: ResNet18
Pretrained weights: None
Feature dimension: 512
Projection dimension: 128
Total parameters: 11,504,832
Trainable parameters: 11,504,832


In [34]:
def nt_xent_loss(
    z1,
    z2,
    temperature=0.2,
):
    batch_size = z1.size(0)

    z1 = nn.functional.normalize(
        z1,
        dim=1,
    )

    z2 = nn.functional.normalize(
        z2,
        dim=1,
    )

    representations = torch.cat(
        [z1, z2],
        dim=0,
    )

    similarity = (
        representations
        @ representations.T
    ) / temperature

    mask = torch.eye(
        2 * batch_size,
        dtype=torch.bool,
        device=similarity.device,
    )

    similarity = similarity.masked_fill(
        mask,
        float("-inf"),
    )

    positive_indices = torch.arange(
        batch_size,
        device=similarity.device,
    )

    targets = torch.cat([
        positive_indices + batch_size,
        positive_indices,
    ])

    loss = nn.functional.cross_entropy(
        similarity,
        targets,
    )

    return loss

In [35]:
simclr_optimizer = torch.optim.Adam(
    simclr_model.parameters(),
    lr=SIMCLR_LR,
    weight_decay=SIMCLR_WEIGHT_DECAY,
)

simclr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    simclr_optimizer,
    T_max=SIMCLR_EPOCHS,
)

print("Optimizer: Adam")
print("Learning rate:", SIMCLR_LR)
print("Weight decay:", SIMCLR_WEIGHT_DECAY)
print("Temperature:", SIMCLR_TAU)

Optimizer: Adam
Learning rate: 0.001
Weight decay: 0.0001
Temperature: 0.2


In [36]:
simclr_history = []
simclr_start = time.time()

for epoch in range(
    1,
    SIMCLR_EPOCHS + 1,
):
    simclr_model.train()

    running_loss = 0.0
    samples_seen = 0

    for batch_index, (
        view1,
        view2,
    ) in enumerate(
        simclr_loader,
        start=1,
    ):
        view1 = view1.to(
            device,
            non_blocking=True,
        )

        view2 = view2.to(
            device,
            non_blocking=True,
        )

        simclr_optimizer.zero_grad(
            set_to_none=True
        )

        _, z1 = simclr_model(view1)
        _, z2 = simclr_model(view2)

        loss = nt_xent_loss(
            z1,
            z2,
            temperature=SIMCLR_TAU,
        )

        if not torch.isfinite(loss):
            raise RuntimeError(
                f"Nonfinite SimCLR loss at "
                f"epoch {epoch}, "
                f"batch {batch_index}"
            )

        loss.backward()
        simclr_optimizer.step()

        batch_size = view1.size(0)

        running_loss += (
            float(loss.detach().cpu())
            * batch_size
        )

        samples_seen += batch_size

    simclr_scheduler.step()

    epoch_loss = (
        running_loss / samples_seen
    )

    simclr_history.append({
        "epoch": epoch,
        "contrastive_loss": epoch_loss,
        "learning_rate": (
            simclr_scheduler
            .get_last_lr()[0]
        ),
    })

    print(
        f"SimCLR epoch "
        f"{epoch:02d}/{SIMCLR_EPOCHS} | "
        f"loss={epoch_loss:.4f} | "
        f"lr="
        f"{simclr_scheduler.get_last_lr()[0]:.6f}"
    )

simclr_training_time = (
    time.time() - simclr_start
)

print()
print(
    "SimCLR pretraining time:",
    f"{simclr_training_time:.2f} seconds",
)

SimCLR epoch 01/15 | loss=4.6858 | lr=0.000989
SimCLR epoch 02/15 | loss=3.5662 | lr=0.000957
SimCLR epoch 03/15 | loss=2.9675 | lr=0.000905
SimCLR epoch 04/15 | loss=2.7361 | lr=0.000835
SimCLR epoch 05/15 | loss=2.6058 | lr=0.000750
SimCLR epoch 06/15 | loss=2.5166 | lr=0.000655
SimCLR epoch 07/15 | loss=2.4492 | lr=0.000552
SimCLR epoch 08/15 | loss=2.3872 | lr=0.000448
SimCLR epoch 09/15 | loss=2.3579 | lr=0.000345
SimCLR epoch 10/15 | loss=2.3094 | lr=0.000250
SimCLR epoch 11/15 | loss=2.2758 | lr=0.000165
SimCLR epoch 12/15 | loss=2.2376 | lr=0.000095
SimCLR epoch 13/15 | loss=2.2196 | lr=0.000043
SimCLR epoch 14/15 | loss=2.2134 | lr=0.000011
SimCLR epoch 15/15 | loss=2.2024 | lr=0.000000

SimCLR pretraining time: 1270.80 seconds


In [37]:
simclr_history_df = pd.DataFrame(
    simclr_history
)

simclr_history_df

,epoch,contrastive_loss,learning_rate
0,1,4.685846,0.000989
1,2,3.566236,0.000957
2,3,2.967535,0.000905
3,4,2.736068,0.000835
4,5,2.605790,0.000750
5,6,2.516636,0.000655
6,7,2.449178,0.000552
7,8,2.387204,0.000448
8,9,2.357939,0.000345
9,10,2.309381,0.000250


In [38]:
simclr_encoder_checkpoint = (
    "checkpoints/simclr_encoder.pt"
)

torch.save(
    {
        "encoder_state_dict":
            simclr_model.encoder.state_dict(),
        "architecture": "resnet18",
        "pretrained": False,
        "feature_dim":
            simclr_model.feature_dim,
        "projection_dim":
            PROJECTION_DIM,
        "temperature":
            SIMCLR_TAU,
        "subset_indices":
            simclr_indices,
        "SID4":
            SID4,
        "SEED":
            SEED,
        "epochs":
            SIMCLR_EPOCHS,
    },
    simclr_encoder_checkpoint,
)

simclr_history_df.to_csv(
    "artifacts/simclr_training_history.csv",
    index=False,
)

print(
    "Saved SimCLR encoder:",
    simclr_encoder_checkpoint,
)

Saved SimCLR encoder: checkpoints/simclr_encoder.pt


In [39]:
simclr_encoder = simclr_model.encoder

for parameter in simclr_encoder.parameters():
    parameter.requires_grad = False

simclr_linear_classifier = nn.Linear(
    simclr_model.feature_dim,
    10,
).to(device)

simclr_encoder = simclr_encoder.to(device)

simclr_encoder_trainable = sum(
    p.numel()
    for p in simclr_encoder.parameters()
    if p.requires_grad
)

simclr_linear_trainable = sum(
    p.numel()
    for p in simclr_linear_classifier.parameters()
    if p.requires_grad
)

print(
    "Trainable encoder parameters:",
    simclr_encoder_trainable,
)

print(
    "Trainable linear classifier parameters:",
    simclr_linear_trainable,
)

assert simclr_encoder_trainable == 0
assert simclr_linear_trainable > 0

print("Freeze check passed.")

Trainable encoder parameters: 0
Trainable linear classifier parameters: 5130
Freeze check passed.


In [40]:
simclr_linear_generator = torch.Generator()
simclr_linear_generator.manual_seed(SEED)

simclr_linear_loader = DataLoader(
    linear_train_subset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    generator=simclr_linear_generator,
)

print(
    "SimCLR linear evaluation labeled images:",
    len(linear_train_subset),
)

assert len(linear_train_subset) == 500

SimCLR linear evaluation labeled images: 500


In [41]:
SIMCLR_LINEAR_EPOCHS = 20
SIMCLR_LINEAR_LR = 1e-3

simclr_linear_criterion = (
    nn.CrossEntropyLoss()
)

simclr_linear_optimizer = torch.optim.Adam(
    simclr_linear_classifier.parameters(),
    lr=SIMCLR_LINEAR_LR,
)

simclr_linear_scheduler = (
    torch.optim.lr_scheduler.CosineAnnealingLR(
        simclr_linear_optimizer,
        T_max=SIMCLR_LINEAR_EPOCHS,
    )
)

In [42]:
SIMCLR_LINEAR_EPOCHS = 20
SIMCLR_LINEAR_LR = 1e-3

simclr_linear_criterion = (
    nn.CrossEntropyLoss()
)

simclr_linear_optimizer = torch.optim.Adam(
    simclr_linear_classifier.parameters(),
    lr=SIMCLR_LINEAR_LR,
)

simclr_linear_scheduler = (
    torch.optim.lr_scheduler.CosineAnnealingLR(
        simclr_linear_optimizer,
        T_max=SIMCLR_LINEAR_EPOCHS,
    )
)

In [43]:
simclr_linear_history = []
simclr_linear_start = time.time()

for epoch in range(
    1,
    SIMCLR_LINEAR_EPOCHS + 1,
):
    simclr_encoder.eval()
    simclr_linear_classifier.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in simclr_linear_loader:
        images = images.to(
            device,
            non_blocking=True,
        )

        labels = labels.to(
            device,
            non_blocking=True,
        )

        simclr_linear_optimizer.zero_grad(
            set_to_none=True
        )

        with torch.no_grad():
            features = simclr_encoder(
                images
            )

        logits = simclr_linear_classifier(
            features
        )

        loss = simclr_linear_criterion(
            logits,
            labels,
        )

        if not torch.isfinite(loss):
            raise RuntimeError(
                f"Nonfinite SimCLR linear "
                f"loss at epoch {epoch}"
            )

        loss.backward()
        simclr_linear_optimizer.step()

        running_loss += (
            float(loss.detach().cpu())
            * labels.size(0)
        )

        predictions = logits.argmax(
            dim=1
        )

        correct += int(
            (
                predictions == labels
            ).sum().item()
        )

        total += labels.size(0)

    simclr_linear_scheduler.step()

    epoch_loss = (
        running_loss / total
    )

    epoch_accuracy = (
        correct / total
    )

    simclr_linear_history.append({
        "epoch":
            epoch,
        "train_loss":
            epoch_loss,
        "train_accuracy":
            epoch_accuracy,
        "learning_rate":
            simclr_linear_scheduler
            .get_last_lr()[0],
    })

    print(
        f"SimCLR linear epoch "
        f"{epoch:02d}/"
        f"{SIMCLR_LINEAR_EPOCHS} | "
        f"loss={epoch_loss:.4f} | "
        f"train_acc="
        f"{epoch_accuracy * 100:.2f}% | "
        f"lr="
        f"{simclr_linear_scheduler.get_last_lr()[0]:.6f}"
    )

simclr_linear_training_time = (
    time.time() - simclr_linear_start
)

print()
print(
    "SimCLR linear training time:",
    f"{simclr_linear_training_time:.2f} seconds",
)

SimCLR linear epoch 01/20 | loss=2.2166 | train_acc=19.80% | lr=0.000994
SimCLR linear epoch 02/20 | loss=1.7219 | train_acc=41.00% | lr=0.000976
SimCLR linear epoch 03/20 | loss=1.4866 | train_acc=49.40% | lr=0.000946
SimCLR linear epoch 04/20 | loss=1.3867 | train_acc=51.20% | lr=0.000905
SimCLR linear epoch 05/20 | loss=1.3164 | train_acc=52.40% | lr=0.000854
SimCLR linear epoch 06/20 | loss=1.2796 | train_acc=55.60% | lr=0.000794
SimCLR linear epoch 07/20 | loss=1.2445 | train_acc=55.60% | lr=0.000727
SimCLR linear epoch 08/20 | loss=1.2235 | train_acc=56.20% | lr=0.000655
SimCLR linear epoch 09/20 | loss=1.2051 | train_acc=57.40% | lr=0.000578
SimCLR linear epoch 10/20 | loss=1.1900 | train_acc=57.40% | lr=0.000500
SimCLR linear epoch 11/20 | loss=1.1727 | train_acc=58.00% | lr=0.000422
SimCLR linear epoch 12/20 | loss=1.1638 | train_acc=59.40% | lr=0.000345
SimCLR linear epoch 13/20 | loss=1.1551 | train_acc=59.20% | lr=0.000273
SimCLR linear epoch 14/20 | loss=1.1481 | train_acc

In [44]:
simclr_encoder.eval()
simclr_linear_classifier.eval()

correct = 0
total = 0
total_loss = 0.0

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(
            device,
            non_blocking=True,
        )

        labels = labels.to(
            device,
            non_blocking=True,
        )

        features = simclr_encoder(
            images
        )

        logits = simclr_linear_classifier(
            features
        )

        loss = simclr_linear_criterion(
            logits,
            labels,
        )

        total_loss += (
            float(loss.detach().cpu())
            * labels.size(0)
        )

        predictions = logits.argmax(
            dim=1
        )

        correct += int(
            (
                predictions == labels
            ).sum().item()
        )

        total += labels.size(0)

simclr_test_loss = (
    total_loss / total
)

simclr_test_accuracy = (
    correct / total
)

print(
    f"SimCLR linear test loss: "
    f"{simclr_test_loss:.4f}"
)

print(
    f"SimCLR linear test accuracy: "
    f"{simclr_test_accuracy * 100:.2f}%"
)

assert np.isfinite(simclr_test_loss)
assert 0 <= simclr_test_accuracy <= 1

SimCLR linear test loss: 1.4622
SimCLR linear test accuracy: 47.42%


In [45]:
abc_comparison = pd.DataFrame([
    {
        "Method":
            "Supervised",
        "Labeled Images":
            500,
        "Unlabeled Pretraining Images":
            0,
        "Test Accuracy":
            test_accuracy,
    },
    {
        "Method":
            "Rotation SSL",
        "Labeled Images":
            500,
        "Unlabeled Pretraining Images":
            100000,
        "Test Accuracy":
            rotation_test_accuracy,
    },
    {
        "Method":
            "SimCLR",
        "Labeled Images":
            500,
        "Unlabeled Pretraining Images":
            20000,
        "Test Accuracy":
            simclr_test_accuracy,
    },
])

abc_comparison

,Method,Labeled Images,Unlabeled Pretraining Images,Test Accuracy
0,Supervised,500,0,0.444000
1,Rotation SSL,500,100000,0.288625
2,SimCLR,500,20000,0.474250


In [46]:
simclr_linear_history_df = pd.DataFrame(
    simclr_linear_history
)

simclr_linear_history_df.to_csv(
    "artifacts/simclr_linear_history.csv",
    index=False,
)

simclr_linear_checkpoint = (
    "checkpoints/simclr_linear_classifier.pt"
)

torch.save(
    {
        "classifier_state_dict":
            simclr_linear_classifier.state_dict(),
        "encoder_checkpoint":
            simclr_encoder_checkpoint,
        "encoder_frozen":
            True,
        "linear_classes":
            10,
        "labeled_indices":
            labeled_indices,
        "test_accuracy":
            simclr_test_accuracy,
        "SID4":
            SID4,
        "SEED":
            SEED,
    },
    simclr_linear_checkpoint,
)

part_c_metrics = {
    "timestamp":
        datetime.now().isoformat(),
    "SID4":
        SID4,
    "SEED":
        SEED,
    "unlabeled_subset_images":
        20000,
    "augmentations": [
        "RandomResizedCrop",
        "RandomHorizontalFlip",
        "ColorJitter",
        "RandomGrayscale",
    ],
    "temperature":
        SIMCLR_TAU,
    "projection_dim":
        PROJECTION_DIM,
    "simclr_epochs":
        SIMCLR_EPOCHS,
    "simclr_final_loss":
        simclr_history[-1][
            "contrastive_loss"
        ],
    "simclr_training_time_seconds":
        simclr_training_time,
    "linear_labeled_images":
        500,
    "linear_epochs":
        SIMCLR_LINEAR_EPOCHS,
    "linear_trainable_encoder_parameters":
        simclr_encoder_trainable,
    "linear_trainable_classifier_parameters":
        simclr_linear_trainable,
    "test_loss":
        simclr_test_loss,
    "test_accuracy":
        simclr_test_accuracy,
    "linear_training_time_seconds":
        simclr_linear_training_time,
}

with open(
    "artifacts/part_c_metrics.json",
    "w",
    encoding="utf8",
) as f:
    json.dump(
        part_c_metrics,
        f,
        indent=2,
    )

print(
    json.dumps(
        part_c_metrics,
        indent=2,
    )
)

print()
print(
    "Saved:",
    simclr_linear_checkpoint,
)

{
  "timestamp": "2026-10-06T19:42:08.362241",
  "SID4": 2046,
  "SEED": 2046,
  "unlabeled_subset_images": 20000,
  "augmentations": [
    "RandomResizedCrop",
    "RandomHorizontalFlip",
    "ColorJitter",
    "RandomGrayscale"
  ],
  "temperature": 0.2,
  "projection_dim": 128,
  "simclr_epochs": 15,
  "simclr_final_loss": 2.2023507356643677,
  "simclr_training_time_seconds": 1270.8013761043549,
  "linear_labeled_images": 500,
  "linear_epochs": 20,
  "linear_trainable_encoder_parameters": 0,
  "linear_trainable_classifier_parameters": 5130,
  "test_loss": 1.4621512269973755,
  "test_accuracy": 0.47425,
  "linear_training_time_seconds": 11.228367567062378
}

Saved: checkpoints/simclr_linear_classifier.pt
